# generate_pairs
Local generation: `stage1.jsonl` -> cleaned `pairs.jsonl`.

In [ ]:
import re, json, os, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import AutoTokenizer
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))
SPLIT = os.environ["SPLIT"]
ROOT  = Path("../../data")
INFILE    = ROOT / "build"  / SPLIT / "stage1.jsonl"
PAIRS_OUT = ROOT / "output" / SPLIT / "pairs.jsonl"
CLANG   = "/opt/homebrew/opt/llvm/bin/clang"   # Homebrew clang 22.1.7
JOBS    = 10
LIMIT   = None        # None = all of stage1

EXPECTED_VER = "22.1.7"
TARGET    = "arm64-apple-macosx26.0.0"
TIMEOUT_S = 10
MAX_TOKENS = 7500

TOK = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-0.5B")

CLANG_ARGS = ["-x", "c", "-", "-S", "-emit-llvm", "-O0",
              "-Xclang", "-disable-O0-optnone", f"--target={TARGET}", "-o", "-",
              "-Wno-implicit-function-declaration"]
O3_ARGS = ["-x", "c", "-", "-S", "-emit-llvm", "-O3",
           f"--target={TARGET}", "-o", "-",
           "-Wno-implicit-function-declaration"]

_LABEL = re.compile(r"^[\w.$-]+:")

def verify_clang(clang):
    out = subprocess.run([clang, "--version"], capture_output=True, text=True).stdout
    if EXPECTED_VER not in out:
        raise RuntimeError(f"clang is NOT pinned {EXPECTED_VER}:\n{out}")
    print("clang OK:", out.splitlines()[0])

def analyze_ir(ir):
    n_instr, has_branch, in_def = 0, False, False
    for raw in ir.splitlines():
        s = raw.strip()
        if s.startswith("define ") and s.endswith("{"):
            in_def = True; continue
        if not in_def: continue
        if s == "}": in_def = False; continue
        if not s or s.startswith(";") or _LABEL.match(s): continue
        n_instr += 1
        if s.startswith(("br ", "switch ", "indirectbr ")): has_branch = True
    return n_instr, has_branch

def clean_c(s):
    return "\n".join(l for l in s.splitlines() if not l.lstrip().startswith("#"))

def clean_ir(s):
    out = []
    for l in s.splitlines():
        st = l.lstrip()
        if st.startswith("attributes #") or st.startswith("!"):
            break
        out.append(l)
    return "\n".join(out).rstrip() + "\n"

def compile_one(rec):
    src = (rec.get("real_deps") or "") + "\n" + rec["func_def"]
    base = {"fname": rec.get("fname"), "path": rec.get("path"), "src": src}
    try:
        p = subprocess.run([CLANG, *CLANG_ARGS], input=src,
                           capture_output=True, text=True, timeout=TIMEOUT_S)
    except subprocess.TimeoutExpired:
        return {**base, "compile_ok": False, "ir_instr_count": None, "has_branch": None}
    if p.returncode != 0:
        return {**base, "compile_ok": False, "ir_instr_count": None, "has_branch": None}
    n, b = analyze_ir(p.stdout)
    return {**base, "compile_ok": True, "ir_instr_count": n, "has_branch": b}

def compile_o3(src):
    try:
        p = subprocess.run([CLANG, *O3_ARGS], input=src,
                           capture_output=True, text=True, timeout=TIMEOUT_S)
    except subprocess.TimeoutExpired:
        return None
    return p.stdout if p.returncode == 0 else None

In [ ]:
# === measure at -O0, gate to survivors (>=30 instr + branch) ===
verify_clang(CLANG)
records = [json.loads(l) for l in INFILE.open() if l.strip()]
records = [r for r in records if r.get("func_def")]
if LIMIT:
    records = records[:LIMIT]
print(f"compiling {len(records)} functions, {JOBS} workers")

results = []
with ThreadPoolExecutor(max_workers=JOBS) as ex:
    futs = [ex.submit(compile_one, r) for r in records]
    for i, fut in enumerate(as_completed(futs), 1):
        results.append(fut.result())
        if i % 1000 == 0: print(f"  {i}/{len(records)}")

survivors = [r for r in results if r["compile_ok"]
             and r["ir_instr_count"] >= 30 and r["has_branch"]]
print(f"{len(survivors)} survivors of {len(results)} ({len(survivors)/len(results):.0%})")


In [ ]:
# === generate CLEANED (C -> -O3 IR) pairs -> pairs.jsonl ===
# cleaning applied HERE, before tokenizing/writing, so the corpus is clean on disk.
PAIRS_OUT.parent.mkdir(parents=True, exist_ok=True)

def make_pair(meta):
    o3 = compile_o3(meta["src"])
    if o3 is None:
        return None
    o3_n, _ = analyze_ir(o3)
    c_clean  = clean_c(meta["src"])
    o3_clean = clean_ir(o3)
    n_tok = len(TOK(c_clean)["input_ids"]) + len(TOK(o3_clean)["input_ids"])
    if n_tok > MAX_TOKENS:
        return None
    return {
        "fname": meta["fname"], "path": meta["path"],
        "c_source": c_clean,
        "o3_ir": o3_clean,
        "o0_instr_count": meta["ir_instr_count"],
        "o3_instr_count": o3_n,
    }

pairs = []
with PAIRS_OUT.open("w") as out, ThreadPoolExecutor(max_workers=JOBS) as ex:
    futs = [ex.submit(make_pair, m) for m in survivors]
    for i, fut in enumerate(as_completed(futs), 1):
        r = fut.result()
        if r:
            pairs.append(r); out.write(json.dumps(r) + "\n")
        if i % 500 == 0: print(f"  {i}/{len(survivors)}")
print(f"wrote {len(pairs)} pairs -> {PAIRS_OUT}")


In [ ]:
# === verify cleaning was baked into the WRITTEN file ===
written = [json.loads(l) for l in PAIRS_OUT.open() if l.strip()]
bad_c  = sum(1 for p in written if any(l.lstrip().startswith("#") for l in p["c_source"].splitlines()))
bad_ir = sum(1 for p in written if "\nattributes #" in p["o3_ir"] or "\n!llvm" in p["o3_ir"])
print("leftover # lines in c_source:", bad_c, "(want 0)")
print("leftover attributes/!llvm in o3_ir:", bad_ir, "(want 0)")
print("all targets have a define:", all("define " in p["o3_ir"] for p in written), "(want True)")
mains = [p for p in written if p["fname"] == "main"]
print("main pairs:", len(mains), "distinct sources:", len(set(p["c_source"] for p in mains)))
